In [1]:
%pip install firebase-admin gspread

  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)

   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------------------------------------- 0/4 [oauthlib]
   ---------- ----------------------------- 1/4 [requests-oauthlib]
   ---------- ----------------------------- 1/4 [requests-oauthlib]
   -------------------- ------------------- 2/4 [google-auth-oauthlib]
   ------------------------------ --------- 3/4 [gspread]
   ---------------------------------------- 4/4 [gspread]

N

The value specified in an AutoRun registry key could not be parsed.


In [2]:
import firebase_admin
from firebase_admin import credentials, firestore
import gspread

# File should match what you uploaded
FIREBASE_KEY_FILE = 'C:\\Users\\mdj20\\Downloads\\watermelon-cup-production-firebase-adminsdk-2rmym-f1467bd3a9.json'

# Firebase init
cred = credentials.Certificate(FIREBASE_KEY_FILE)
firebase_admin.initialize_app(cred)
db = firestore.client()

# GSpread init
gc = gspread.service_account(filename=FIREBASE_KEY_FILE)


In [3]:
sheet_id = '1mLXtTPr8FTzGZQKDZJnuQoAqVtreMzus3GFnFRcBEGc'
worksheet_name = '2026_Updating_Roster'

In [6]:
def export_registered_users_to_sheet(sheet_id, worksheet_name="Sheet1"):
    users_ref = db.collection('users')
    docs = users_ref.where('registered2026', '==', True).stream()

    user_dicts = []
    for doc in docs:
        raw = doc.to_dict()
        cleaned = {}
        for k, v in raw.items():
            if hasattr(v, 'isoformat'):  # Timestamp
                cleaned[k] = v.isoformat()
            elif isinstance(v, list):    # Convert lists to comma-separated strings
                cleaned[k] = ', '.join(str(item) for item in v)
            else:
                cleaned[k] = v
        user_dicts.append(cleaned)

    if not user_dicts:
        print("No users with registered2026 == true.")
        return

    sheet = gc.open_by_key(sheet_id)
    worksheet = sheet.worksheet(worksheet_name)

    headers = sorted(set().union(*(d.keys() for d in user_dicts)))
    rows = [headers]
    for user in user_dicts:
        row = [user.get(h, '') for h in headers]
        rows.append(row)

    worksheet.clear()
    worksheet.update(rows)

    print(f"✅ Exported {len(user_dicts)} users to '{worksheet_name}' tab in Google Sheet.")


In [7]:
# call the export function
export_registered_users_to_sheet(sheet_id, worksheet_name)

c:\Users\mdj20\Documents\GitHub\watermelon-cup\.venv\Lib\site-packages\google\cloud\firestore_v1\base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


✅ Exported 72 users to '2026_Updating_Roster' tab in Google Sheet.
